### Create Connection with the API

In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS youtube_earthquake_dashboard_conn
TYPE HTTP
OPTIONS (
  host = 'https://earthquake.usgs.gov',
  port = '443',
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

### Create a base url from connection

In [0]:
%py
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn = w.connections.get("youtube_earthquake_dashboard_conn")
base_path = f"{conn.options['host']}{conn.options['base_path']}"

### Make catalog and schema name dynamic

In [0]:
dbutils.widgets.text('catalog_name', 'youtube_earthquake_dashboard', 'youtube_earthquake_dashboard')
catalog_name = dbutils.widgets.get('catalog_name')

dbutils.widgets.text("schema_name", "bronze", "bronze")
schema_name = dbutils.widgets.get("schema_name")

### Make the volume folder under catalog bronze schema

In [0]:
volume_name = "volume_youtube_earthquake_dashboard"
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

### Check the reponse of the API and Save into the Folder

In [0]:
import requests
import json
import datetime

url = f"{base_path}summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
  raise Exception(f"Error: {response.status_code} - {response.text}")
data = response.json()
dateformate = datetime.datetime.now().strftime("%Y-%m-%d")
filename = f"youtube_earthquake_dashboard_bronze_{dateformate}.json"
#/Volumes/project_1_youtube_dev/bronze/volume_project_1_dev
dbutils.fs.put(f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{filename}", json.dumps(data), overwrite=True)